# Reproducibility 분석 최종본 — GRNBoost2 vs SpaGTL

## 이 노트북의 목적
GRN 추론 결과(motif-pruning **이후**의 최종 regulon, `expanded_post_pruning/run*_motif_pruned_edges_expanded.tsv`)가
**10번의 반복 실행(run)** 사이에서 얼마나 재현되는지를 검증합니다. 추가로, post-pruning 재현성이 낮게 나오는 원인을
**pruning 이전(raw co-expression) 단계와 비교**하여 규명합니다.

- **SpaGRN은 단일 run**이라 재현성 분석의 대상이 아닙니다 (재현성 평가는 여러 run이 있어야 가능).
- 메인 분석은 **post-pruning 최종 regulon** 기준입니다 (논문의 모든 생물학적 주장이 여기 기반). 다만 재현성 저하의
  원인을 규명하기 위해 pre-pruning(raw co-expression, motif filtering 이전) 데이터도 **비교 목적으로** 별도 섹션에서 사용합니다.

## 세 개의 질문

| | 질문 | 비교 대상 |
|---|---|---|
| **Q1** | 전체적으로 run 간에 얼마나 흔들리는가? (전반적 안정성) | 전체 edge/TF, 필터 없음 |
| **Q1x (신규)** | 그 흔들림은 pruning 이전부터 있었나, pruning이 증폭한 것인가? | pre-pruning vs post-pruning |
| **Q2** | 흔들려도, **반복해서 검출되는 core**끼리는 서로 논리적으로 일관되는가? (외부 정답과 비교 아님 — 자기 자신과의 일관성) | occurrence≥3/5/10로 걸러낸 부분집합만 |

## 지표 요약표 (전체 미리보기)

| 구분 | 지표 | 무엇을 재는가 | 판단 기준 |
|---|---|---|---|
| Q1a | mean, SD, CV% | edge/TF/target **개수**의 run간 변동 | CV%가 낮을수록 안정 |
| Q1b | Spearman ρ (k-sweep) | top-k 안 공통 edge의 importance **순위** 일치도 | ρ=1 완전일치, ρ=0 무관(무작위 기준) |
| Q1b | Top-k Jaccard (k-sweep) | 상위 k개 edge **집합** 자체의 일치도 | 무작위 기대값(permutation) 대비 |
| Q1b | WJS (union-weighted Jaccard) | 집합 일치 + **weight 유사성**까지 반영 (Kang et al. 2021 스타일) | Jaccard와 같은 방향이면 결과 강건 |
| Q1x (신규) | pre vs post Top-k Jaccard | pruning 전후 안정성 비교 | pre>>post면 "pruning이 증폭"을 시사 |
| Q2b | target-set Jaccard fold, **median[IQR]** | occurrence 통과 TF들의 target 구성 일치도, universe 크기로 보정 | fold=1이면 무작위와 동일, 클수록 유의미 |
| Q2c(보충) | GO brain-relevance % | occurrence 높은 core의 생물학적 그럴듯함 | 참고용(주관적 키워드 판정) |


In [ ]:
import glob, re, ast, itertools
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import spearmanr
plt.rcParams['font.family']='AppleGothic'; plt.rcParams['axes.unicode_minus']=False
plt.rcParams['figure.dpi']=110

BASE = '/Users/kimchaeyoung/Desktop/thesis_data'
METHOD_DIR = {'GRNBoost2':'d0', 'SpaGTL':'spagtl'}
COLORS = {'GRNBoost2':'#2a78d6', 'SpaGTL':'#eb6834'}
THRESHOLDS = [3, 5, 10]
rng = np.random.default_rng(0)

def load_run_edges(method, run):
    p = f"{BASE}/{METHOD_DIR[method]}/multi_run/multi_run_analysis/expanded_post_pruning/run{run}_motif_pruned_edges_expanded.tsv"
    return pd.read_csv(p, sep='\t')

def load_cons(method, k):
    p = f"{BASE}/{METHOD_DIR[method]}/multi_run/multi_run_analysis/occurrence_consensus/motif_pruned_edges_occurrence_ge_{k:02d}.tsv"
    return pd.read_csv(p, sep='\t')

RUNS = {m: {r: load_run_edges(m, r) for r in range(1, 11)} for m in METHOD_DIR}
for m in METHOD_DIR:
    print(m, '10개 run 로드 완료, 예시 run1 edge 수 =', len(RUNS[m][1]))


---
# Q1a. Run-to-run 네트워크 크기 변동

## 질문
같은 방법을 10번 반복 실행했을 때, 결과 네트워크의 **규모(edge/TF/target 개수)** 자체가 얼마나 들쭉날쭉한가?

## 지표: mean, SD, CV(%)
$$\text{CV} = \frac{\text{SD}}{\text{mean}} \times 100\%$$

- `mean`, `SD` = 10개 run의 개수 분포의 평균과 표준편차
- **CV(변동계수)**를 쓰는 이유: SD 자체는 방법마다 개수 스케일(GRNBoost2 ~26,000 edges vs SpaGTL ~7,000 edges)이 달라 그대로 비교하면 불공정합니다.
  CV는 평균 대비 상대적 비율이라 **스케일이 다른 두 방법을 공정하게 비교**할 수 있습니다.
- 판단 기준: 절대적인 "좋다/나쁘다" 기준은 없지만(엄밀한 법칙은 아님), 관용적으로 CV가 낮을수록(대략 10% 이하) 안정적, 높을수록(20% 이상) 변동이 크다고 서술합니다.


In [ ]:
rows = []
for m in METHOD_DIR:
    for r, d in RUNS[m].items():
        rows.append({'method': m, 'run': r, 'n_edges': len(d), 'n_TF': d['TF'].nunique(), 'n_target': d['target'].nunique()})
sizedf = pd.DataFrame(rows)

panel_titles = {'n_edges': 'Edges', 'n_TF': 'TFs', 'n_target': 'Targets'}
methods = list(METHOD_DIR)

fig, axes = plt.subplots(1, 3, figsize=(12, 4.5))
for ax, col in zip(axes, ['n_edges', 'n_TF', 'n_target']):
    data = [sizedf[sizedf['method'] == m][col].values for m in methods]
    bp = ax.boxplot(
        data, tick_labels=methods, patch_artist=True, widths=0.75,
        boxprops=dict(linewidth=1.2, edgecolor='#3a3a3a'),
        medianprops=dict(color='#3a3a3a', linewidth=2),
        whiskerprops=dict(color='#3a3a3a', linewidth=1.2),
        capprops=dict(color='#3a3a3a', linewidth=1.2),
        flierprops=dict(marker='o', markerfacecolor='#8a8a8a', markeredgecolor='none', markersize=5, alpha=0.6),
    )
    for patch, m in zip(bp['boxes'], methods):
        patch.set_facecolor(COLORS[m])
        patch.set_alpha(0.85)
    ax.set_title(panel_titles[col], fontsize=12, fontweight='bold', pad=8)
    ax.set_xlim(0.5, 2.5)
    ax.margins(y=0.08)
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    ax.grid(axis='y', color='#e2e2e2', linewidth=0.8, zorder=0)
    ax.set_axisbelow(True)
    ax.tick_params(labelsize=10)

fig.suptitle("Q1a. Run-to-run network size variability (10 runs each)", fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

summary = sizedf.groupby('method')[['n_edges', 'n_TF', 'n_target']].agg(['mean', 'std']).round(1)
cv = sizedf.groupby('method')[['n_edges', 'n_TF', 'n_target']].apply(lambda g: g.std() / g.mean() * 100).round(1)
print(summary)
print("\nCV (%):")
print(cv)


**그림 설명**
- x축: 방법(GRNBoost2, SpaGTL) / y축: 각 패널(edge 개수, TF 개수, target 개수)
- 검은 점: 10개 run 각각의 실제 값 (분포 확인용) / 상자: IQR / 텍스트: mean, SD, CV%

## 결론
- edge 개수 CV: GRNBoost2 8% vs SpaGTL 20% → SpaGTL이 훨씬 불안정
- TF 개수 CV: GRNBoost2 2% vs SpaGTL 19% → **TF 개수만 보면 GRNBoost2가 압도적으로 안정적**
- target 개수 CV: GRNBoost2 2% vs SpaGTL 13%
- **모든 지표에서 SpaGTL이 GRNBoost2보다 변동폭이 큼** — 이는 Q1b, Q1c에서도 일관되게 재확인됨.


---
# Q1b. Edge 내용의 재현성 — Spearman(순위) vs Jaccard(집합) vs WJS(집합+weight), 모두 k-sweep

## 질문
"중요하다"고 뽑힌 상위 k개 edge들이 (1) 공통으로 뽑힌 것끼리는 순위가 비슷한가, (2) 애초에 같은 edge가 다시 뽑히기는 하는가,
(3) 그 둘을 합쳐 "집합 일치 + weight 유사성"까지 한 번에 반영하면 어떤가? **모두 k(top-k 크기)를 sweep**하여 상위로 갈수록 재현성이 어떻게 변하는지 봅니다.

## 지표 ①: Top-k Spearman's rank correlation
$$\rho_{\text{rank}}(k) = \text{Spearman}\big(\text{importance ranking of } A_k \cap B_k \text{ in run } r_1,\ \text{same in } r_2\big)$$
공통 edge만 골라 순위 상관만 봅니다. **함정**: "무엇이 뽑히는가"의 불안정성은 반영 못함. 무작위 기준선: ρ→0.

## 지표 ②: Top-k edge Jaccard index
$$J_k(A, B) = \frac{|A_k \cap B_k|}{|A_k \cup B_k|}$$
"어떤 edge가 선택되는가"(집합 자체)만 봅니다 — 순위는 안 봄. 무작위 기준선: permutation(그 방법의 전체 edge universe에서 크기 k 무작위 2세트, 30회 반복).

## 지표 ③: Weighted Jaccard Similarity (WJS) — Kang et al. (2021, *Frontiers in Genetics*) 스타일
$$WJS(A,B) = \frac{\sum_{e \in A \cup B} \min(w_{e,A}, w_{e,B})}{\sum_{e \in A \cup B} \max(w_{e,A}, w_{e,B})}$$
- $A \cup B$ = 두 run의 top-k 합집합. 한쪽에만 있는 edge는 weight 0으로 처리 → **집합 불일치와 weight 불일치를 동시에 벌점(penalize)**.
- GRNBoost2와 SpaGTL은 importance의 절대 스케일이 다르므로(예: GRNBoost2는 tree importance, SpaGTL은 attention 기반 weight),
  **run 안에서 percentile-rank(0~1)로 정규화한 뒤** WJS를 계산합니다.
- 이 지표는 Kang et al. (2021)이 6개 scRNA-seq 네트워크 추론 방법의 재현성을 평가할 때 실제로 사용한 지표로,
  Jaccard(perINT)만으로는 못 보는 "겹치는 edge들의 weight까지 비슷한가"를 추가로 검증합니다.

## 왜 세 지표를 같이 봐야 하는가
Spearman만 보면 SpaGTL이 더 안정적으로 보이는 **착시**가 생길 수 있습니다. Jaccard와 WJS로 "애초에 같은 게 뽑히는가"를 봐야 진짜 그림이 보입니다.


In [ ]:
def edge_imp_map(d):
    return dict(zip(map(tuple, d[['TF','target']].values), d['importance'].values))

def topk_spearman_mean(runs, k):
    names = list(runs)
    maps = {n: edge_imp_map(v.nlargest(k, 'importance')) for n, v in runs.items()}
    vals = []
    for i in range(len(names)):
        for j in range(i+1, len(names)):
            a, b = maps[names[i]], maps[names[j]]
            common = set(a) & set(b)
            if len(common) < 10: continue
            va = [a[e] for e in common]; vb = [b[e] for e in common]
            rho = spearmanr(va, vb).correlation
            if rho is not None and not np.isnan(rho):
                vals.append(rho)
    return (np.mean(vals), np.std(vals)) if vals else (np.nan, np.nan)

def topk_jaccard_mean(runs, k):
    names = list(runs)
    topsets = {n: set(map(tuple, v.nlargest(k,'importance')[['TF','target']].values)) for n, v in runs.items()}
    vals = []
    for i in range(len(names)):
        for j in range(i+1, len(names)):
            A, B = topsets[names[i]], topsets[names[j]]
            u = A | B
            if u: vals.append(len(A & B)/len(u))
    return np.mean(vals), np.std(vals)

def expected_topk_jaccard(k, univ_size, n_perm=30):
    idx = np.arange(univ_size)
    vals = []
    for _ in range(n_perm):
        a = set(rng.choice(idx, size=min(k, univ_size), replace=False))
        b = set(rng.choice(idx, size=min(k, univ_size), replace=False))
        u = a | b
        if u: vals.append(len(a & b)/len(u))
    return np.mean(vals)

def to_topk_pctrank_map(d, k):
    top = d.nlargest(k, 'importance').copy()
    top['pct_rank'] = top['importance'].rank(pct=True)
    return dict(zip(map(tuple, top[['TF','target']].values), top['pct_rank'].values))

def wjs(mapA, mapB):
    all_edges = set(mapA) | set(mapB)
    num, den = 0.0, 0.0
    for e in all_edges:
        wa = mapA.get(e, 0.0); wb = mapB.get(e, 0.0)
        num += min(wa, wb); den += max(wa, wb)
    return num/den if den>0 else np.nan

def topk_wjs_mean(runs, k):
    names = list(runs)
    maps = {n: to_topk_pctrank_map(v, k) for n, v in runs.items()}
    vals = []
    for i in range(len(names)):
        for j in range(i+1, len(names)):
            vals.append(wjs(maps[names[i]], maps[names[j]]))
    return np.mean(vals), np.std(vals)

print("함수 정의 완료")


In [ ]:
ks_common = [50,100,200,300,500,750,1000,1500,2000,3000,5000,7500,10000,15000,20000]
univ_size_edges = {}
for m in METHOD_DIR:
    all_edges = set()
    for r in range(1,11):
        all_edges |= set(zip(RUNS[m][r]['TF'], RUNS[m][r]['target']))
    univ_size_edges[m] = len(all_edges)
    print(f"{m} edge universe(10-run union) size = {len(all_edges)}")

fig, axes = plt.subplots(1, 3, figsize=(19, 6))

# --- panel 1: Spearman sweep ---
ax = axes[0]
sp_summary = []
for m in METHOD_DIR:
    max_k = min(len(v) for v in RUNS[m].values())
    ks = [k for k in ks_common if k <= max_k]
    ys, sds = [], []
    for k in ks:
        mu, sd = topk_spearman_mean(RUNS[m], k)
        ys.append(mu); sds.append(sd)
        sp_summary.append({'method':m,'k':k,'mean_rho':mu})
    ys, sds = np.array(ys), np.array(sds)
    ax.plot(ks, ys, 'o-', color=COLORS[m], label=f'{m}', markersize=4)
    ax.fill_between(ks, ys-sds, ys+sds, color=COLORS[m], alpha=0.15)
ax.axhline(0, color='red', linestyle='--', linewidth=1.3, label='random (rho=0)')
ax.set_xscale('log'); ax.set_xlabel('Top-k edges (log)'); ax.set_ylabel("Spearman rho (common edges within top-k)")
ax.set_title("① Spearman\n(rank agreement, common edges)")
ax.legend(fontsize=8); ax.grid(alpha=0.3, which='both')

# --- panel 2: Top-k Jaccard sweep ---
ax = axes[1]
topk_summary = []
for m in METHOD_DIR:
    max_k = min(len(v) for v in RUNS[m].values())
    ks = [k for k in ks_common if k <= max_k]
    ys, sds = [], []
    for k in ks:
        mu, sd = topk_jaccard_mean(RUNS[m], k)
        ys.append(mu); sds.append(sd)
        topk_summary.append({'method':m,'k':k,'mean_jaccard':mu})
    ys, sds = np.array(ys), np.array(sds)
    ax.plot(ks, ys, 'o-', color=COLORS[m], label=f'{m} (observed)', markersize=4)
    ax.fill_between(ks, ys-sds, ys+sds, color=COLORS[m], alpha=0.15)
    exp_vals = [expected_topk_jaccard(k, univ_size_edges[m]) for k in ks]
    ax.plot(ks, exp_vals, '--', color=COLORS[m], alpha=0.5, linewidth=1.3, label=f'{m} (random)')
ax.set_xscale('log'); ax.set_xlabel('Top-k edges (log)'); ax.set_ylabel('mean pairwise Jaccard')
ax.set_title("② Top-k Jaccard\n(set agreement, dashed=random)")
ax.legend(fontsize=8); ax.grid(alpha=0.3, which='both')

# --- panel 3: WJS sweep ---
ax = axes[2]
wjs_summary = []
ks_wjs = [100,500,1000,2000,3000,5000]
for m in METHOD_DIR:
    max_k = min(len(v) for v in RUNS[m].values())
    ks = [k for k in ks_wjs if k <= max_k]
    ys, sds = [], []
    for k in ks:
        mu, sd = topk_wjs_mean(RUNS[m], k)
        ys.append(mu); sds.append(sd)
        wjs_summary.append({'method':m,'k':k,'mean_wjs':mu})
    ys, sds = np.array(ys), np.array(sds)
    ax.plot(ks, ys, 'o-', color=COLORS[m], label=f'{m}', markersize=4)
    ax.fill_between(ks, ys-sds, ys+sds, color=COLORS[m], alpha=0.15)
ax.set_xscale('log'); ax.set_xlabel('Top-k edges (log)'); ax.set_ylabel('mean pairwise WJS')
ax.set_title("③ WJS (Kang et al. 2021 style)\n(set + weight agreement)")
ax.legend(fontsize=8); ax.grid(alpha=0.3, which='both')

fig.suptitle("Q1b. Three views of the SAME top-k edges, all swept across k", fontsize=12)
plt.tight_layout(); plt.show()

print(pd.DataFrame(sp_summary).pivot(index='k', columns='method', values='mean_rho').round(3))
print()
print(pd.DataFrame(topk_summary).pivot(index='k', columns='method', values='mean_jaccard').round(3))
print()
print(pd.DataFrame(wjs_summary).pivot(index='k', columns='method', values='mean_wjs').round(3))


**그림 설명**
- **①Spearman**: x축=top-k(log), y축=공통 edge 순위 상관. 빨간 점선=무작위(ρ=0).
- **②Top-k Jaccard**: x축=top-k(log), y축=집합 일치도. 점선=그 방법 고유 무작위 기준선.
- **③WJS**: x축=top-k(log), y축=percentile-rank 정규화 후 weighted Jaccard (Kang et al. 2021 지표).

## 결론
- **Spearman**: SpaGTL이 전 k 구간에서 GRNBoost2보다 높거나 비슷함 → 이것만 보면 SpaGTL이 더 안정적으로 오해.
- **Jaccard**: SpaGTL(k=1000에서 0.08)이 GRNBoost2(0.28)보다 훨씬 낮고 k를 늘려도 평평함.
- **WJS**: Jaccard와 **거의 동일한 패턴** (k=100까지는 SpaGTL이 근소 우세, k≥500부터 SpaGTL 급락, GRNBoost2는 상승) →
  weight 유사성을 추가로 반영해도 결론이 바뀌지 않아, **Jaccard 기반 결론이 강건함(robust)**이 확인됨.
- **해석**: SpaGTL은 "어떤 edge를 뽑을지는 극도로 불안정하지만, 우연히 공통으로 뽑힌 소수의 edge끼리는 순위·weight가 일관"됩니다.
  Spearman만 보면 이 불안정성을 놓치므로, **재현성 평가의 주 지표는 Jaccard/WJS(집합 일치) 계열로 삼아야 합니다.**


---
# Q1x (신규). Pre-pruning vs Post-pruning — SpaGTL 재현성 저조의 원인 규명

## 배경
Q1b에서 SpaGTL의 post-pruning(motif-pruning 이후 최종 regulon) top-k Jaccard가 GRNBoost2보다 훨씬 낮다는 걸 확인했습니다.
이 저조함의 원인은 최소 세 가지 가설로 나뉩니다:

1. **(a) Pruning 증폭 가설**: pre-pruning(raw co-expression)에서는 안정적인데, motif pruning이라는 **hard threshold** 단계가
   작은 importance 값 차이를 "포함/제외"라는 이분법적 차이로 증폭시킨다.
2. **(b) 알고리즘 자체 불안정 가설**: pre-pruning부터 이미 불안정하다 — 즉 SpaGTL의 fine-tuning/추론 자체가 이 real dataset에서 잘 수렴하지 않는다.
3. **(c) 특정 run 이상치 가설**: 10개 run 중 특정 하나(예: run1)가 technical failure 등으로 유독 이상해서 전체 평균을 끌어내린다.

이를 구분하기 위해 **pre-pruning(raw co-expression) 데이터**(`spagtl_QK_edges.tsv`, `step1/adjacencies.tsv`, 각 run당 ~130만 edge)로
동일한 Top-k Jaccard 분석을 반복하고, leave-one-run-out으로 특정 run의 영향도 검증합니다.

**주의**: 이 섹션에서만 예외적으로 pre-pruning(raw) 데이터를 사용합니다 — 목적은 "최종 결과의 재현성을 재평가"하는 게 아니라
"post-pruning 재현성 저하의 원인이 어디서 발생하는지"를 진단하는 것이므로, 노트북 서두에서 밝힌 "메인 분석은 post-pruning 기준"이라는
원칙과 배치되지 않습니다.


In [ ]:
def load_pre_grn(run):
    return pd.read_csv(f"{BASE}/d0/multi_run/run{run}/step1/adjacencies.tsv", sep='\t')

def load_pre_spagtl(run):
    return pd.read_csv(f"{BASE}/spagtl/multi_run/run{run}/spagtl_QK_edges.tsv", sep='\t')

PRE = {
    'GRNBoost2': {r: load_pre_grn(r) for r in range(1, 11)},
    'SpaGTL': {r: load_pre_spagtl(r) for r in range(1, 11)},
}
for m in PRE:
    print(f"{m} pre-pruning run1 edge 수 = {len(PRE[m][1])}")


In [ ]:
ks_cmp = [100, 500, 1000, 2000, 5000, 10000]

pre_rows, post_rows = [], []
for m in METHOD_DIR:
    for k in ks_cmp:
        mu_pre, sd_pre = topk_jaccard_mean(PRE[m], k)
        pre_rows.append({'method':m,'k':k,'jaccard':mu_pre,'stage':'pre-pruning'})
        max_k_post = min(len(v) for v in RUNS[m].values())
        mu_post, sd_post = topk_jaccard_mean(RUNS[m], min(k, max_k_post))
        post_rows.append({'method':m,'k':k,'jaccard':mu_post,'stage':'post-pruning'})

cmpdf = pd.concat([pd.DataFrame(pre_rows), pd.DataFrame(post_rows)])

fig, axes = plt.subplots(1, 2, figsize=(12, 5.5), sharey=True)
for ax, m in zip(axes, METHOD_DIR):
    sub = cmpdf[cmpdf['method']==m]
    for stage, style in [('pre-pruning','o--'), ('post-pruning','o-')]:
        s = sub[sub['stage']==stage].sort_values('k')
        ax.plot(s['k'], s['jaccard'], style, color=COLORS[m], label=stage,
                alpha=(0.5 if stage=='pre-pruning' else 1.0), markersize=5)
    ax.set_xscale('log'); ax.set_xlabel('Top-k edges (log)')
    ax.set_title(m); ax.legend(fontsize=9)
axes[0].set_ylabel('mean pairwise Top-k Jaccard')
fig.suptitle("Q1x. Pre-pruning vs Post-pruning Top-k Jaccard\n(dashed=raw co-expression, solid=motif-pruned final regulon)", fontsize=12)
plt.tight_layout(); plt.show()

print(cmpdf.pivot_table(index='k', columns=['method','stage'], values='jaccard').round(3))


In [ ]:
# leave-one-run-out: run1 (또는 다른 run) anomaly 여부 검증 (post-pruning SpaGTL, k=1000)
k = 1000
def topk_jaccard_mean_subset(runs_dict, k, run_ids):
    topsets = {n: set(map(tuple, runs_dict[n].nlargest(k,'importance')[['TF','target']].values)) for n in run_ids}
    vals = []
    for i in range(len(run_ids)):
        for j in range(i+1, len(run_ids)):
            A, B = topsets[run_ids[i]], topsets[run_ids[j]]
            u = A | B
            if u: vals.append(len(A & B)/len(u))
    return np.mean(vals)

baseline = topk_jaccard_mean_subset(RUNS['SpaGTL'], k, list(range(1,11)))
print(f"SpaGTL post-pruning, k={k}, 전체 10-run 기준 Jaccard = {baseline:.3f}\n")
print("run 하나씩 제외했을 때 (leave-one-run-out):")
loo_rows = []
for r_out in range(1, 11):
    remain = [r for r in range(1,11) if r != r_out]
    mu = topk_jaccard_mean_subset(RUNS['SpaGTL'], k, remain)
    loo_rows.append({'run_excluded': r_out, 'jaccard_without': mu, 'diff_vs_baseline': mu - baseline})
    print(f"  run{r_out} 제외: {mu:.3f} ({mu-baseline:+.3f})")

loodf = pd.DataFrame(loo_rows)

fig, ax = plt.subplots(figsize=(7,4.5))
ax.bar(loodf['run_excluded'], loodf['diff_vs_baseline'], color=COLORS['SpaGTL'])
ax.axhline(0, color='gray', linewidth=1)
ax.set_xlabel('제외한 run 번호'); ax.set_ylabel('전체 10-run 기준 대비 Jaccard 변화')
ax.set_title(f'SpaGTL post-pruning k={k}: leave-one-run-out\n(어떤 run 하나를 빼도 크게 달라지지 않으면 특정 run의 문제가 아님)')
ax.set_xticks(range(1,11))
plt.tight_layout(); plt.show()


**그림 설명**
- **Pre vs Post 비교 그림**: x축=top-k(log), y축=Top-k Jaccard. 점선=pre-pruning(raw), 실선=post-pruning(최종 regulon).
- **Leave-one-run-out 그림**: x축=제외한 run 번호, y축=그 run을 뺐을 때 전체 기준 대비 Jaccard 변화량. 0 근처에 고르게 분포하면 특정 run의 문제가 아님.

## 결론 — 가설 (a) 확정, (c) 기각

**Pre-pruning 결과 (놀라운 반전)**: pre-pruning 단계에서는 **SpaGTL이 GRNBoost2보다 오히려 더 안정적**입니다
(예: k=1000에서 SpaGTL 0.346 vs GRNBoost2 0.248). 그런데 **post-pruning에서는 정반대로 뒤집혀** SpaGTL이 0.077, GRNBoost2가 0.284가 됩니다.

**Leave-one-run-out 결과**: run1을 포함해 어떤 run 하나를 빼도 Jaccard가 거의 변하지 않습니다(변화량이 모두 ±0.01 이내) —
즉 특정 run 하나의 technical failure가 원인이 아니라, **10개 run 전체에 걸쳐 고르게 나타나는 구조적 현상**입니다.

**종합 판단**:
- ✅ **가설 (a) 채택**: SpaGTL은 raw 단계에서는 안정적인 신호를 만들지만, **motif pruning이라는 hard-threshold 단계가
  작은 importance 값의 미세한 차이를 "포함/제외"라는 이분법적 차이로 증폭**시켜, 최종 결과의 재현성을 크게 떨어뜨립니다.
- ❌ **가설 (b) 기각**: SpaGTL의 추론/fine-tuning 자체는 이 데이터에서 불안정하지 않습니다(오히려 pre-pruning 기준으로는 GRNBoost2보다 안정적).
- ❌ **가설 (c) 기각**: 특정 run의 이상치 문제가 아닙니다.

**중요한 함의**: 이 결과는 "SpaGTL이 근본적으로 불안정한 방법"이라는 단순한 결론을 뒤집습니다. 오히려 **SpaGTL + motif pruning의 조합이
문제**이며, 이는 (i) SpaGTL의 importance 분포가 GRNBoost2보다 더 평평해서(edge 간 값 차이가 작아서) 임계값 근처에서 순위가 쉽게 뒤바뀌거나,
(ii) motif pruning 파라미터가 GRNBoost2용으로 최적화되어 SpaGTL 출력엔 덜 적합할 가능성을 시사합니다 (후속 검증 필요, 본 논문 범위 밖).


---
# Q1c. TF(레귤론) 정체성 vs 개별 target edge의 재현율

## 질문
"이 TF가 조절자다"라는 판단과 "이 TF가 정확히 이 유전자를 조절한다"는 판단 중 어느 쪽이 더 잘 재현되는가?

## 지표: occurrence 임계값별 누적 생존율(%)
- 각 TF(혹은 edge)가 10개 run 중 **몇 개에서 등장했는지**(occurrence, 1~10)를 셉니다.
- $k=1,\dots,10$ 전 구간에 대해 "occurrence $\ge k$인 TF/edge가 전체의 몇 %인가"를 계산해 누적곡선으로 그립니다.

## 참고: 이 지표는 Q1a/Q1b와 무엇이 다른가
- Q1a는 "개수"(크기)만 보고, Q1b는 "edge 내용"(top-k, 전체 스펙트럼)을 보는 반면,
- Q1c는 **"TF 계층 vs edge 계층"이라는 재현성의 구조적 차이**를 봅니다.


In [ ]:
edge_occ_count, tf_occ_count = {}, {}
for m in METHOD_DIR:
    ec, tf_runs = {}, {}
    for r in range(1, 11):
        df = RUNS[m][r]
        for e in zip(df['TF'], df['target']):
            ec[e] = ec.get(e, 0) + 1
        for tf in df['TF'].unique():
            tf_runs.setdefault(tf, set()).add(r)
    edge_occ_count[m] = pd.Series(list(ec.values()))
    tf_occ_count[m] = pd.Series([len(v) for v in tf_runs.values()])

fig, axes = plt.subplots(1, 2, figsize=(11, 5), sharey=True)
for ax, m in zip(axes, METHOD_DIR):
    ks = list(range(1, 11))
    edge_cum = [100*(edge_occ_count[m]>=k).sum()/len(edge_occ_count[m]) for k in ks]
    tf_cum = [100*(tf_occ_count[m]>=k).sum()/len(tf_occ_count[m]) for k in ks]
    ax.plot(ks, tf_cum, 'o-', color=COLORS[m], label=f'TF (regulon), n={len(tf_occ_count[m])}', linewidth=2.5)
    ax.plot(ks, edge_cum, 's--', color=COLORS[m], label=f'Edge, n={len(edge_occ_count[m])}', alpha=0.6, linewidth=2.5)
    for k, v in zip(ks, tf_cum):
        if k in (1,3,5,10): ax.annotate(f"{v:.0f}%", (k,v), textcoords="offset points", xytext=(0,6), ha='center', fontsize=8)
    for k, v in zip(ks, edge_cum):
        if k in (1,3,5,10): ax.annotate(f"{v:.0f}%", (k,v), textcoords="offset points", xytext=(0,-12), ha='center', fontsize=8)
    ax.set_xlabel("occurrence threshold k\n(detected in >=k out of 10 runs)")
    ax.set_title(m); ax.set_xticks(ks); ax.legend(fontsize=8, loc='upper right'); ax.grid(alpha=0.3)
axes[0].set_ylabel("% of TFs / edges meeting the threshold\n(cumulative, higher = more reproducible)")
fig.suptitle("Q1c. TF identity survives repetition far better than\nthe specific target edges attached to it", fontsize=12)
plt.tight_layout(); plt.show()

for m in METHOD_DIR:
    print(f"{m}: TF occurrence=10: {100*(tf_occ_count[m]>=10).sum()/len(tf_occ_count[m]):.0f}%, Edge occurrence=10: {100*(edge_occ_count[m]>=10).sum()/len(edge_occ_count[m]):.0f}%")


**그림 설명**
- x축: occurrence 임계값 k(1~10) / y축: 그 임계값을 넘는 TF(실선) 혹은 edge(점선)의 비율(%)

## 결론
At an occurrence threshold of at least five of ten runs, 52% of GRNBoost2 TFs and 15% of GRNBoost2 edges survived,
compared with 29% of SpaGTL TFs and 3% of SpaGTL edges. At the strict 10-of-10 threshold, 22% of GRNBoost2 TFs and 2%
of its edges survived, whereas 4% of SpaGTL TFs and fewer than approximately 0.5% of its edges survived. Looking at
the opposite end of the spectrum, 56% of GRNBoost2's edges and 75% of SpaGTL's edges appeared in only a single run
(occurrence=1) out of the ten -- i.e., the majority of inferred edges, especially for SpaGTL, are essentially
one-off, non-reproducible calls.

This difference is partly definitional: a recurrent TF requires only one associated target in a run, whereas a
recurrent edge requires the exact same TF-target pair. **결론적으로, 두 방법 모두 "TF 정체성(누가 조절자인가)"은
비교적 재현되지만, "그 TF가 정확히 어떤 유전자를 조절하는지(target 구성)"는 대부분의 edge가 단 한 번의 run에서만
등장하는 수준으로 크게 흔들립니다.** SpaGTL은 이 두 축(TF 생존율, edge 생존율) 모두에서 GRNBoost2보다 일관되게 더
가파르게 무너지며, 이는 Q1x에서 확인한 "motif pruning 증폭 효과"와 일맥상통합니다.


---
# Q2b. 반복 검출된 TF들의 target 구성 일치도 (universe 크기로 보정, median[IQR] 기준)

## 질문
Q1에서 "전체적으로 흔들린다"는 걸 확인했다면, 이번엔 **"그럼에도 반복해서 뽑히는 core(occurrence 높은 TF)들끼리는 서로 논리적으로 일관되는가"**를 봅니다.
이는 **외부 정답(마커 유전자 등)과 비교하는 게 아니라, 그 방법 자신의 결과와 자기 자신을 비교하는 자기 일관성(self-consistency) 검증**입니다.

## 왜 occurrence 기준이고 top-k 기준이 아닌가
- Q1b의 top-k는 "**한 run 안에서** importance 순위가 높은가"라는, 재현성과 무관한 독립적 기준이라, 이 기준으로 하면 Q1b와 결과가 중복됩니다.
- 여기서 쓰는 occurrence≥k는 "**10개 run에 걸쳐** 몇 번 등장했는가"라는, **재현성 그 자체로 정의된 부분집합**입니다.

## 지표: target-set Jaccard, universe-corrected fold
- occurrence≥3(또는 5, 10)를 만족하는 TF 하나를 고르고, 그 TF가 등장한 모든 run 쌍에 대해 **target 유전자 집합**끼리 Jaccard를 계산 후 평균.
- **universe 크기로 보정**: 그 TF의 평균 target 개수만큼 그 방법의 전체 target universe에서 무작위 두 집합을 뽑아(50회 반복) 기대 Jaccard를 구하고,
  $$\text{fold} = \frac{\text{실제 Jaccard}}{\text{무작위 기대 Jaccard}}$$

## Median[IQR]을 메인으로 쓰는 이유 (기존 mean에서 변경)
fold 값의 분포를 확인한 결과, **모든 조건에서 skewness가 1.8~5.4로 강한 오른쪽 꼬리(long right tail)** 분포입니다
(예: GRNBoost2 occurrence≥10은 skewness=5.39, 최대값 1279배 vs 중앙값 65배 — 극소수 TF가 극단적으로 높은 fold를 보여 mean을 크게 끌어올립니다).
이런 분포에서 **mean은 소수 outlier TF에 과도하게 좌우**되므로, **median[IQR]을 메인 지표로, mean은 참고용 부록으로** 병기합니다.
(raw Jaccard 값 자체는 부록 표 `TABLE_Q2b_target_jaccard_by_threshold.csv`에 남겨둡니다.)


In [ ]:
tf_occ = {}
for m in METHOD_DIR:
    tf_occ[m] = {}
    for r in range(1, 11):
        for tf in RUNS[m][r]['TF'].unique():
            tf_occ[m][tf] = tf_occ[m].get(tf, 0) + 1
run_tf_targets = {m: {r: RUNS[m][r].groupby('TF')['target'].apply(set).to_dict() for r in range(1, 11)} for m in METHOD_DIR}
universe_targets = {m: sorted(set().union(*[set(RUNS[m][r]['target']) for r in range(1, 11)])) for m in METHOD_DIR}
for m in METHOD_DIR:
    print(f"{m} target universe(10-run union) size = {len(universe_targets[m])}")

def expected_jaccard(n1, n2, univ_size, n_perm=50):
    idx = np.arange(univ_size)
    vals = np.empty(n_perm)
    for i in range(n_perm):
        a = set(rng.choice(idx, size=min(n1, univ_size), replace=False))
        b = set(rng.choice(idx, size=min(n2, univ_size), replace=False))
        u = a | b
        vals[i] = len(a & b)/len(u) if u else np.nan
    return np.nanmean(vals), np.nanstd(vals)

jaccard_rows, corrected_rows = [], []
for m in METHOD_DIR:
    univ_size = len(universe_targets[m])
    for k in THRESHOLDS:
        tfs_at_k = [tf for tf, occ in tf_occ[m].items() if occ >= k]
        for tf in tfs_at_k:
            pair_j, pair_sizes = [], []
            for r1, r2 in itertools.combinations(range(1, 11), 2):
                s1 = run_tf_targets[m][r1].get(tf); s2 = run_tf_targets[m][r2].get(tf)
                if s1 is not None and s2 is not None:
                    u = s1 | s2
                    if u:
                        pair_j.append(len(s1 & s2)/len(u))
                        pair_sizes.append((len(s1), len(s2)))
            if not pair_j: continue
            real_mean = np.mean(pair_j)
            jaccard_rows.append({'method':m,'occurrence_ge':k,'TF':tf,'mean_target_jaccard':real_mean})
            n1_avg = int(np.mean([p[0] for p in pair_sizes])); n2_avg = int(np.mean([p[1] for p in pair_sizes]))
            exp_mean, exp_sd = expected_jaccard(n1_avg, n2_avg, univ_size, n_perm=50)
            fold = real_mean/exp_mean if exp_mean>0 else np.nan
            z = (real_mean-exp_mean)/exp_sd if exp_sd>0 else np.nan
            corrected_rows.append({'method':m,'occurrence_ge':k,'TF':tf,'real_jaccard':real_mean,
                                    'expected_random_jaccard':exp_mean,'fold_vs_random':fold,'z_vs_random':z})

jdf = pd.DataFrame(jaccard_rows)
ucdf = pd.DataFrame(corrected_rows)

print("=== fold 분포 skewness (median vs mean 판단 근거) ===")
for m in METHOD_DIR:
    for k in THRESHOLDS:
        sub = ucdf[(ucdf['method']==m)&(ucdf['occurrence_ge']==k)]['fold_vs_random'].dropna()
        print(f"{m} occurrence>={k}: n={len(sub)}, median={sub.median():.1f}, mean={sub.mean():.1f}, skew={sub.skew():.2f}")

print()
print(ucdf.groupby(['method','occurrence_ge'])['fold_vs_random'].agg(['median','mean','std']).round(2))


In [ ]:
occ_ks = list(range(1, 11))
trend_rows = []
for m in METHOD_DIR:
    for k in occ_ks:
        tfs_at_k = [tf for tf, occ in tf_occ[m].items() if occ == k]
        pair_means = []
        for tf in tfs_at_k:
            pair_j = []
            for r1, r2 in itertools.combinations(range(1, 11), 2):
                s1 = run_tf_targets[m][r1].get(tf); s2 = run_tf_targets[m][r2].get(tf)
                if s1 is not None and s2 is not None:
                    u = s1 | s2
                    if u: pair_j.append(len(s1 & s2)/len(u))
            if pair_j: pair_means.append(np.mean(pair_j))
        trend_rows.append({'method': m, 'occurrence_eq': k, 'n_TF': len(tfs_at_k),
                            'mean_jaccard': np.mean(pair_means) if pair_means else np.nan})
trend_df = pd.DataFrame(trend_rows)
# occurrence==1인 TF는 그 TF가 등장한 run이 하나뿐이라 비교할 run 짝이 없어 Jaccard가 정의되지 않음 -> 그래프에서 제외
plot_df = trend_df.dropna(subset=['mean_jaccard'])

fig, ax = plt.subplots(figsize=(9.5, 5.5))
for m in METHOD_DIR:
    sub = plot_df[plot_df['method'] == m].sort_values('occurrence_eq')
    ax.plot(sub['occurrence_eq'], sub['mean_jaccard'], 'o-', color=COLORS[m], label=m, markersize=7, linewidth=2.2)

# 라벨은 시리즈 고유색이 아니라 "그 지점에서 위/아래" 기준으로 배치 -> 두 선이 교차해도 안 겹침
wide = plot_df.pivot(index='occurrence_eq', columns='method', values='mean_jaccard')
for k, row in wide.iterrows():
    vals = row.dropna().sort_values()
    if len(vals) < 2:
        continue
    lo_m, hi_m = vals.index[0], vals.index[1]
    ax.annotate(f"{row[lo_m]:.3f}", (k, row[lo_m]), textcoords='offset points', xytext=(0, -20),
                ha='center', fontsize=12, color=COLORS[lo_m], fontweight='bold')
    ax.annotate(f"{row[hi_m]:.3f}", (k, row[hi_m]), textcoords='offset points', xytext=(0, 12),
                ha='center', fontsize=12, color=COLORS[hi_m], fontweight='bold')

ax.set_xticks(range(2, 11))
ax.set_xlabel('TF occurrence (exactly k of 10 runs)', fontsize=12, fontweight='bold')
ax.set_ylabel('Mean within TF-Target set Jaccard', fontsize=12, fontweight='bold')
ax.set_ylim(-0.02, 0.28)
ax.legend(frameon=False, loc='upper left', fontsize=11)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
ax.grid(axis='y', color='#e2e2e2', linewidth=0.8, zorder=0)
ax.set_axisbelow(True)
ax.tick_params(labelsize=11)
fig.suptitle('Raw (uncorrected) mean target-set Jaccard, by exact TF occurrence count\n(occurrence=1 omitted: no second run to compare against)', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

print(trend_df.pivot(index='occurrence_eq', columns='method', values=['n_TF', 'mean_jaccard']).round(3))


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 5.5), sharey=True)
for ax, m in zip(axes, METHOD_DIR):
    sub = ucdf[ucdf['method']==m]
    data = [sub[sub['occurrence_ge']==k]['fold_vs_random'].dropna().values for k in THRESHOLDS]
    positions = list(range(1, len(THRESHOLDS)+1))
    bp = ax.boxplot(data, tick_labels=[f">={k}\n(n TFs={len(d)})" for k,d in zip(THRESHOLDS,data)], patch_artist=True)
    for patch in bp['boxes']: patch.set_facecolor(COLORS[m])
    for pos, d in zip(positions, data):
        med = np.median(d); mean = np.mean(d)
        ax.text(pos, np.percentile(d,95), f"median={med:.0f}x\n(mean={mean:.0f}x)", ha='center', va='bottom', fontsize=8)
    ax.set_yscale('log'); ax.axhline(1, color='gray', linestyle='--', linewidth=1)
    ax.set_title(m); ax.set_xlabel("occurrence threshold")
axes[0].set_ylabel("target-set Jaccard fold-enrichment vs random\n(log scale; dashed=1 -> no better than chance)")
fig.suptitle("Q2b. Among TFs repeatedly detected (occurrence-based, NOT top-k),\ndo their target gene sets agree beyond random chance? (MEDIAN as main statistic)", fontsize=11)
plt.tight_layout(); plt.show()


**그림 설명**
- x축: occurrence 임계값(≥3, ≥5, ≥10), 괄호 안 n TFs = 그 임계값을 만족하는 TF의 개수
- y축: 무작위 대비 fold-enrichment (log scale), 회색 점선=1(무작위와 동일)
- 텍스트: **median을 메인으로 표기**, mean은 괄호 안 참고용

## 결론 (median 기준)
- GRNBoost2: **median fold ≈ 65~73배** (occurrence 임계값에 거의 무관하게 안정적으로 높음; mean은 103~109배로 outlier 영향에 더 부풀려짐)
- SpaGTL: **median fold ≈ 30~43배** (임계값이 오를수록 median도 함께 상승; mean은 45~74배)
- **둘 다 무작위(fold=1)보다 수십 배 이상 높아 "반복 검출되는 core는 우연이 아니라 진짜 일관된 신호"**임이 확인됨 (median 기준으로도 견고).
- 다만 GRNBoost2가 SpaGTL보다 median fold가 대체로 더 높아, universe 크기 차이를 감안해도 GRNBoost2가 더 일관적.
- occurrence 임계값이 오를수록(≥3→≥5→≥10) SpaGTL의 median fold 상승폭이 더 큼 — "엄격하게 걸러낼수록 core의 질이 개선된다"는 정도는 SpaGTL이 더 뚜렷.


---
# Q2c (보충/정성적). GO enrichment로 본 core의 생물학적 그럴듯함

## 목적 — 주의: 이건 정답과 비교하는 지표가 아닙니다
Q2b가 "반복 검출되는 core가 통계적으로 무작위가 아니다"를 보였다면, 이 섹션은 **"그 core가 뭔가 무의미한 잡음이 아니라 생물학적으로 그럴듯한 방향을 가리키는지"**를
정성적으로만 확인하는 **보충 자료**입니다.

## 지표: brain-relevant GO term 비율(%) — 이 지표는 주관적입니다
- occurrence≥3/5/10 core의 target 유전자 목록으로 g:Profiler GO:BP enrichment 실행 (background = 전체 측정 유전자)
- 유의미한 GO term 중 "brain-relevant"로 판정되는 비율을 계산하는데, **이 판정 자체가 제가 정한 키워드 정규식**에 의존하므로 **객관적 지표가 아닙니다.**
- 따라서 **메인 결론(Q1, Q2b)에는 사용하지 않으며**, 참고용으로만 남깁니다.


In [ ]:
from gprofiler import GProfiler
gp = GProfiler(return_dataframe=True)
BRAIN = re.compile('neuron|neural|nervous|synap|axon|dendrit|glia|glial|myelin|forebrain|hindbrain|midbrain|cortex|cortical|hippocamp|cerebell|brain|gliogenesis|neurogenesis|neurotransmitter|neurite|gabaergic|glutamatergic', re.I)
bg = set()
for m in METHOD_DIR:
    for r in range(1, 11):
        bg |= set(RUNS[m][r]['target'])
BG = sorted(bg)

go_rows = []
for m in METHOD_DIR:
    for k in THRESHOLDS:
        df = load_cons(m, k)
        targets = sorted(df['target'].unique())
        try:
            r = gp.profile(organism='mmusculus', query=[str(x) for x in targets], sources=['GO:BP'],
                            background=BG, domain_scope='custom', user_threshold=0.05, no_evidences=True)
        except Exception as e:
            print(m, k, "FAILED", e); continue
        n_sig = len(r)
        n_brain = r['name'].str.contains(BRAIN).sum() if n_sig > 0 else 0
        go_rows.append({'method': m, 'occurrence_ge': k, 'n_target': len(targets), 'n_sig_GO': n_sig,
                         'n_brain_GO': n_brain, 'pct_brain': 100*n_brain/n_sig if n_sig > 0 else np.nan})
godf = pd.DataFrame(go_rows)

fig, ax = plt.subplots(figsize=(7, 5))
for m in METHOD_DIR:
    sub = godf[godf['method']==m].sort_values('occurrence_ge')
    ax.plot(sub['occurrence_ge'], sub['pct_brain'], marker='o', label=m, color=COLORS[m])
    for _, row in sub.iterrows():
        ax.annotate(f"{row['pct_brain']:.0f}%", (row['occurrence_ge'], row['pct_brain']), textcoords="offset points", xytext=(0,8), ha='center', fontsize=8)
ax.set_xticks(THRESHOLDS)
ax.set_xlabel("occurrence threshold (>=k out of 10 runs)")
ax.set_ylabel("% of significant GO:BP terms matching a\nbrain-related keyword regex (subjective)")
ax.set_title("Q2c. SUPPLEMENTARY / qualitative only -\nnot an objective metric (keyword choice is subjective)")
ax.legend()
plt.tight_layout(); plt.show()


**그림 설명**
- x축: occurrence 임계값 / y축: 유의미 GO:BP term 중 brain-relevant 키워드에 걸리는 비율(%) — **주관적 지표임을 다시 강조**

## 결론(참고용)
- 두 방법 모두 occurrence 임계값이 오를수록 brain-relevant 비율이 상승 (GRNBoost2 21→33%, SpaGTL 28→35%)
- 이는 "엄격하게 걸러낸 core일수록 생물학적으로 더 그럴듯한 방향을 가리킨다"는 정황을 보여주지만, **키워드 판정 기준의 주관성 때문에 정량적 결론으로는 사용하지 않습니다.**


---
# 최종 종합 결론

| 질문 | 핵심 지표 | GRNBoost2 | SpaGTL | 판정 |
|---|---|---|---|---|
| Q1a (크기 변동) | CV% | 2~8% | 13~20% | GRNBoost2 안정 |
| Q1b (edge 내용) | Top-k Jaccard/WJS (k=1000) | 0.28 / 0.26 | 0.08 / 0.09 | GRNBoost2 안정 (Spearman만 보면 반대로 오해 가능) |
| Q1x (pre vs post) | pre-pruning Top-k Jaccard (k=1000) | 0.248 | **0.346 (SpaGTL이 더 높음)** | pruning 이전엔 SpaGTL이 오히려 안정적 |
| Q1c (TF vs edge) | occurrence=10 생존율 | TF 22% / edge 2% | TF 4% / edge 0% | 둘 다 TF>edge, GRNBoost2가 전반적으로 더 높음 |
| Q2b (core 일관성) | median fold vs random | 65~73x | 30~43x | 둘 다 무작위 아님, GRNBoost2가 더 높음 |

## 종합 판단

1. **Post-pruning(최종 결과) 기준으로는 GRNBoost2가 Q1과 Q2 양쪽 모두에서 SpaGTL보다 재현성이 높습니다.**
2. **그러나 이 차이의 상당 부분은 SpaGTL 알고리즘 자체의 근본적 결함이 아니라, motif pruning 단계에서 증폭된 것입니다(Q1x)** —
   pre-pruning 기준으로는 SpaGTL이 오히려 GRNBoost2보다 안정적이었습니다. 이는 SpaGTL을 "본질적으로 불안정한 방법"으로 단순화해 서술하면 안 되는 중요한 이유입니다.
3. **SpaGTL도 완전히 무의미한 잡음은 아닙니다** — Q2b에서 무작위 대비 median 30배 이상 높은 fold를 보여, "반복 검출되는 core"는 우연이 아닌 진짜 신호입니다.
4. **핵심 방법론적 함정**: Spearman만 보면 SpaGTL이 더 안정적이라고 오해할 수 있으므로, 재현성 평가에는 반드시 Jaccard/WJS(집합 일치) 계열 지표를 함께 봐야 합니다.
5. **SpaGRN은 단일 run**이라 이 재현성 분석의 대상이 아니며, 이는 세 방법 비교의 공정성 측면에서 명시적 한계로 남습니다.

## 문헌과의 비교 — Kang et al. (2021, *Frontiers in Genetics*)

Kang, Thieffry & Cantini (2021)는 GENIE3, GRNBoost2 등 6개 scRNA-seq 네트워크 추론 방법의 재현성을 **독립적인 두 데이터셋**(같은 생물학적 조건,
다른 시료) 사이의 perINT(percentage of intersection)와 WJS로 평가했습니다. 이들의 핵심 결과:
- GRNBoost2는 top-K=1,000~100처럼 **엄격한 threshold에서 상대적으로 더 좋은 성능**을 보임 (인체 망막: perINT 41.1%, WJS 0.25)
- 가장 좋은 방법조차 perINT 26~54%, WJS 0.1~0.3 수준으로 **완벽한 재현성과는 거리가 있었음**

**본 연구 결과와의 대응**: 본 연구의 GRNBoost2 top-1000 Jaccard 0.284는 `|A∩B|/|A∪B| = J`, `|A∩B|/K = 2J/(1+J)` 관계식으로 환산하면
**약 44% intersection**에 해당하여, Kang et al.의 26~54% perINT 범위와 **대략적인 스케일이 유사**합니다.

**다만 직접적인 수치 비교는 부적절합니다** — 두 연구는 다음이 근본적으로 다릅니다:
- Kang et al.: **독립적인 두 데이터셋**(다른 시료, 다른 환자) 간 비교 / 본 연구: **동일 데이터에서의 stochastic reproducibility**(같은 입력, 알고리즘 자체의 확률성만 다름)
- Kang et al.: 대부분 **pruning 이전(raw weighted network)** / 본 연구: **motif-pruned 최종 regulon**
- metric의 정확한 정의, gene universe 크기도 다름

그럼에도 다음과 같은 **문헌과 정합적인 결론**은 안전하게 내릴 수 있습니다:

> GRNBoost2 showed higher relative repeatability than SpaGTL in our post-pruning analysis, but its absolute
> edge-level repeatability remained moderate, consistent with previous reports that even leading GRN inference
> methods recover only partially overlapping networks across repeated or independent analyses (Kang et al., 2021).

이렇게 서술하면 본 연구의 결과를 과장하지 않으면서도 기존 문헌과 자연스럽게 연결됩니다.
